In [5]:
print("MM21B030")

MM21B030


In [ ]:
from seq2seq_model import Seq2SeqModel 
from load_data import get_data_loaders 
import torch
import torch.nn as nn

# Initialize data loaders and model
train_loader, dev_loader, test_loader, char_to_idx_devanagari, char_to_idx_latin = get_data_loaders()
input_vocab_size = len(char_to_idx_devanagari)
output_vocab_size = len(char_to_idx_latin)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model = Seq2SeqModel(
    input_vocab_size=input_vocab_size,
    output_vocab_size=output_vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    cell_type="gru",
    num_layers_encoder=2,
    num_layers_decoder=2,
    dropout=0.2,
    device=device,
    beam_size=2
).to(device)

criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    
    for src, trg in train_loader:
        # Move data to device
        src = src.to(device)
        trg = trg.to(device)
        
        # Forward pass
        output = model(src, trg[:, :-1])  # Teacher forcing with shifted target
        
        # Calculate loss (ignore padding)
        loss = criterion(output.reshape(-1, output_vocab_size), 
                        trg[:, 1:].reshape(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}')

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for src, trg in dev_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg[:, :-1])
            loss = criterion(output.reshape(-1, output_vocab_size), 
                          trg[:, 1:].reshape(-1))
            val_loss += loss.item()
        print(f'Validation Loss: {val_loss/len(dev_loader):.4f}')

Epoch 1, Loss: 0.5874
Validation Loss: 0.3214
Epoch 2, Loss: 0.2850
Validation Loss: 0.2688


KeyboardInterrupt: 